# AI-XRay · 04 — Comparación de runs, registro y cierre del ciclo

**Clase 4 · Laboratorio Práctico Dirigido — Proyecto Final Independiente**

### Objetivo de este notebook
1. Comparar todos los runs registrados en el Experiment `AI-XRay` de MLflow.
2. Elegir el mejor run con criterio explícito (no solo accuracy).
3. Registrarlo en el **Model Registry** y promoverlo a `Staging`.
4. Exportarlo como archivo local (`models/aixray_model.keras`) para que la API (`api/`) lo sirva sin depender de un servidor de MLflow corriendo.
5. Cerrar el ciclo: cargar el modelo ya registrado y correr una inferencia de ejemplo sobre imágenes del set de test.
6. Checklist final de autoevaluación antes de escribir el Executive Summary.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import mlflow
import mlflow.keras
from mlflow.tracking import MlflowClient

from src.config import MLFLOW_EXPERIMENT_NAME, MLFLOW_MODEL_NAME, MLFLOW_TRACKING_URI, MODELS_DIR, SPLIT_MANIFEST_PATH
from src.preprocessing import dataset_from_manifest

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)


<Experiment: artifact_location='file:///c:/Users/juand/GitHub/M3-Cientifico-Datos-IA-Aplicada-DevSeniorCode/03_ia_aplicada_databricks_arquitecturas/clase_04_laboratorio_practico/AI-XRay/notebooks/mlruns/1', creation_time=1786923858434, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786923858434, lifecycle_stage='active', name='AI-XRay', tags={}, trace_location=None, workspace='default'>

## 1. Comparar los runs del experimento

In [2]:
runs_df = mlflow.search_runs(order_by=["metrics.f1_score DESC"])
columnas_interes = [
    c for c in runs_df.columns
    if c.startswith("metrics.") or c.startswith("params.") or c in ["run_id", "tags.mlflow.runName"]
]
runs_df[columnas_interes]


,run_id,metrics.accuracy,metrics.auc,metrics.val_accuracy,metrics.loss,metrics.f1_score,metrics.recall,metrics.precision,metrics.val_loss,params.fine_tune_at_layer,params.architecture,params.epochs_max,params.seed,params.class_weight,params.batch_size,params.learning_rate,params.optimizer,params.fine_tune,params.dropout,tags.mlflow.runName
0,bea5a2fd6c2249fea314dc6b41ef8357,0.934379,0.982914,0.936877,0.191441,0.933798,0.887417,0.985294,0.205085,143,ResNet50_fine_tuning,3,42,"{0: 1.0014285714285713, 1: 0.9985754985754985}",dynamic,1e-05,Adam,True,0.3,resnet50_finetuning_lr1e-5
1,6c5c909e114b4f17b7888d94f328bad0,0.907989,0.981766,0.913621,0.229946,0.916129,0.940397,0.893082,0.187514,None,ResNet50_frozen,3,42,"{0: 1.0014285714285713, 1: 0.9985754985754985}",dynamic,0.001,Adam,False,0.3,resnet50_frozen_lr1e-3_do0.3
2,f1b2b3ecab5f467c962b431e7e7b3245,0.755350,0.953157,0.880399,0.504362,0.872340,0.814570,0.938931,0.358416,None,ResNet50_frozen,3,42,"{0: 1.0014285714285713, 1: 0.9985754985754985}",dynamic,0.0001,Adam,False,0.5,resnet50_frozen_lr1e-4_do0.5
3,76efcc6f005144f99f7386f9009e6c39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,ResNet50_frozen,15,42,"{0: 1.0014285714285713, 1: 0.9985754985754985}",dynamic,0.001,Adam,False,0.3,resnet50_frozen_lr1e-3_do0.3
4,6562aaa5af804783ac1f086a07171d73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,ResNet50_frozen,15,42,"{0: 1.0014285714285713, 1: 0.9985754985754985}",dynamic,0.001,Adam,False,0.3,resnet50_frozen_lr1e-3_do0.3


**Criterio de selección:** no elegimos el run con mayor `accuracy` a secas -- en un
problema médico con clases desbalanceadas, priorizamos **recall de la clase PNEUMONIA**
(no queremos dejar pasar casos positivos) sin que **precision** ni **F1** se desplomen.
Por eso se ordena por `f1_score`, que balancea ambas cosas, y se revisa `recall` del mejor
candidato antes de decidir.

In [3]:
mejor_run = runs_df.iloc[0]
print("Mejor run:", mejor_run.get("tags.mlflow.runName"))
print("run_id:", mejor_run["run_id"])
print("F1-score:", mejor_run.get("metrics.f1_score"))
print("Recall:", mejor_run.get("metrics.recall"))
print("Precision:", mejor_run.get("metrics.precision"))
print("AUC:", mejor_run.get("metrics.auc"))
print("Accuracy:", mejor_run.get("metrics.accuracy"))


Mejor run: resnet50_finetuning_lr1e-5
run_id: bea5a2fd6c2249fea314dc6b41ef8357
F1-score: 0.9337979094076655
Recall: 0.8874172185430463
Precision: 0.9852941176470589
AUC: 0.9829139072847681
Accuracy: 0.9343794584274292


## 2. Registrar el mejor modelo en el Model Registry y promoverlo a `Staging`

> **Nota:** `transition_model_version_stage` (stages `None`/`Staging`/`Production`/`Archived`)
> es la API que ya se usó en la Clase 3 y la que sigue funcionando en Databricks Free
> Edition, así que la mantenemos aquí por consistencia. MLflow reciente marca esta API
> como *deprecated* a favor de "aliases" (`MlflowClient().set_registered_model_alias(...)`,
> sin el concepto de *stage* fijo) -- si tu versión de MLflow ya no soporta stages,
> reemplaza esta celda por `set_registered_model_alias(MLFLOW_MODEL_NAME, "staging", registered_model.version)` y ajusta `api/model_service.py` para cargar por alias en vez de por stage.

In [4]:
model_uri = f"runs:/{mejor_run['run_id']}/model"
registered_model = mlflow.register_model(model_uri=model_uri, name=MLFLOW_MODEL_NAME)
print(f"Modelo registrado: {registered_model.name} — versión {registered_model.version}")

client = MlflowClient()
client.transition_model_version_stage(
    name=MLFLOW_MODEL_NAME,
    version=registered_model.version,
    stage="Staging",
    archive_existing_versions=False,
)
print(f"Versión {registered_model.version} promovida a Staging ✅")


Successfully registered model 'aixray_pneumonia_classifier'.
2026/08/16 20:07:14 WARNING mlflow.tracking._model_registry.fluent: Run with id bea5a2fd6c2249fea314dc6b41ef8357 has no artifacts at artifact path 'model', registering model based on models:/m-23477c0ea96846b69b6f95a72369b993 instead


Modelo registrado: aixray_pneumonia_classifier — versión 1
Versión 1 promovida a Staging ✅


Created version '1' of model 'aixray_pneumonia_classifier'.
C:\Users\juand\AppData\Local\Temp\ipykernel_38356\1595865049.py:6: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


## 3. Exportar el modelo para que la API lo sirva sin depender de MLflow corriendo

In [5]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
best_model = mlflow.keras.load_model(f"models:/{MLFLOW_MODEL_NAME}/Staging")
export_path = MODELS_DIR / "aixray_model.keras"
best_model.save(export_path)
print("Modelo exportado a:", export_path)
print("La API (api/model_service.py) ya está configurada para cargar este archivo por defecto.")


Modelo exportado a: C:\Users\juand\GitHub\M3-Cientifico-Datos-IA-Aplicada-DevSeniorCode\03_ia_aplicada_databricks_arquitecturas\clase_04_laboratorio_practico\AI-XRay\models\aixray_model.keras
La API (api/model_service.py) ya está configurada para cargar este archivo por defecto.


## 4. Cerrando el ciclo: inferencia sobre imágenes reales del set de test

In [13]:
split = pd.read_csv(SPLIT_MANIFEST_PATH)
test_ds = dataset_from_manifest(split, "test", batch_size=8)

x_batch, y_batch = next(iter(test_ds))
preds = best_model.predict(x_batch, verbose=0).ravel()

for i in range(min(8, len(preds))):
    real = "PNEUMONIA" if y_batch.numpy()[i] == 1 else "NORMAL"
    pred = "PNEUMONIA" if preds[i] >= 0.5 else "NORMAL"
    marca = "✅" if real == pred else "❌"
    print(f"{marca} real={real:10s} predicho={pred:10s} probabilidad={preds[i]:.3f}")


✅ real=NORMAL     predicho=NORMAL     probabilidad=0.001
✅ real=PNEUMONIA  predicho=PNEUMONIA  probabilidad=0.763
✅ real=NORMAL     predicho=NORMAL     probabilidad=0.001
✅ real=NORMAL     predicho=NORMAL     probabilidad=0.000
✅ real=NORMAL     predicho=NORMAL     probabilidad=0.000
✅ real=PNEUMONIA  predicho=PNEUMONIA  probabilidad=0.978
✅ real=NORMAL     predicho=NORMAL     probabilidad=0.005
✅ real=PNEUMONIA  predicho=PNEUMONIA  probabilidad=0.623


Así es exactamente como la API (`api/model_service.py`) usa el modelo en producción:
sin conocer el código de entrenamiento, solo cargando el archivo exportado y aplicando el
mismo preprocesamiento (`preprocess_input` + resize 224×224).

## 5. Checklist final de autoevaluación

Antes de pasar al Executive Summary, confirma que tu proyecto cumple:

- [ ] Usé PySpark (aunque sea el paso simbólico del manifiesto) para el procesamiento de datos.
- [ ] Entrené **al menos 2 variantes** del modelo y las comparé con criterio explícito (no solo accuracy).
- [ ] Cada run quedó registrado en MLflow con parámetros, métricas, artefactos y el modelo.
- [ ] Registré el mejor modelo en el Model Registry y lo promoví a `Staging`.
- [ ] La API (`api/main.py`) responde correctamente en `/health` y `/predict` con el modelo exportado.
- [ ] Revisé la sección de fuga de datos (`data/README.md`) y la voy a mencionar en Limitaciones.
- [ ] Mi Executive Summary incluye el aviso de que este sistema no es una herramienta de diagnóstico médico.

### Cierre
Con el modelo registrado, exportado y sirviendo desde la API, el sistema está completo de
punta a punta. El siguiente paso es escribir el Executive Summary
(`../executive_summary/plantilla_executive_summary.docx`).